# Notebook 08 — Gene Regulatory Networks and Boolean Models

**Module:** 12 — Systems and Network Biology  
**Tier:** 2 — Working competence  
**Notebook:** 08 of 12  
**Time estimate:** 80 minutes

> A gene regulatory network is a directed, signed graph. A Boolean network
> is its simplest dynamical model: each gene is either ON or OFF, and
> trajectories converge to attractors that correspond to cell states.

---
## Step 1 — Motivation

Knowing the PPI network topology is not enough to predict cell behavior — we need
to model *dynamics*. Boolean networks (Kauffman 1969) are the simplest dynamical
model of GRNs. Despite their simplicity (binary states, discrete time), they reproduce
key features of cell biology: stable cell states, bistability, and oscillations.

---
## Step 2 — Intuition

**Boolean network:** Each gene $g_i$ is in state $x_i \in \{0, 1\}$ (OFF/ON).
At each time step, every gene's state is updated according to a Boolean function
of its regulators:
$$x_i(t+1) = f_i(x_{r_1}(t), x_{r_2}(t), \ldots)$$

The function $f_i$ encodes the regulatory logic: AND, OR, NOT, combinations thereof.

**Attractor:** A state (or cycle) the network eventually reaches and stays in.
- **Fixed-point attractor:** $\mathbf{x}(t+1) = \mathbf{x}(t)$ — stable cell state (e.g., differentiated cell)
- **Limit cycle:** a repeating sequence — e.g., the cell cycle

**Basin of attraction:** All initial states that converge to a given attractor.
Large basins correspond to robust, stable cell fates.

**Kauffman hypothesis:** Attractors correspond to cell types. A human with ~200
cell types has ~200 attractors in its gene regulatory dynamics.

---
## Step 3 — Biological Background

**Canonical GRN example: EMT (Epithelial–Mesenchymal Transition)**
- Cancer cells undergo EMT to gain invasive properties
- Key TFs: SNAI1/2, ZEB1/2, TWIST (mesenchymal); E-cadherin, EpCAM (epithelial)
- Mutual repression between epithelial and mesenchymal TFs creates bistability
- Boolean models reproduce: epithelial state, mesenchymal state, and hybrid E/M state

**The lac operon as a Boolean circuit:**
- lacI: 1 if glucose present, else 0
- crp: 1 if glucose absent AND cAMP high, else 0
- lacZYA: 1 if (lactose present AND NOT lacI) AND crp — Boolean AND/NOT logic

**Cell cycle Boolean model (Li et al. 2004):**
- 11-node Boolean network of yeast cell cycle regulators
- 7 of 11 attractors correspond to known cell cycle phases
- Basin of G1 state covers 86% of all initial conditions — robust toward G1

**Synchronous vs. asynchronous updates:**
- **Synchronous:** all genes update simultaneously — deterministic, may not be biologically realistic
- **Asynchronous:** random update order each step — captures stochastic gene expression;
  limit cycles can become attractors

---
## Step 4 — Mathematical Explanation

**State space:** $n$ genes → $2^n$ possible states. For $n=10$: 1024 states.

**State transition graph (STG):** Directed graph with one node per state, edge from
$\mathbf{x}$ to $\mathbf{x}' = \mathbf{f}(\mathbf{x})$. Every node has out-degree exactly 1
(synchronous deterministic); the STG is a functional graph — every path eventually
enters a cycle.

**Number of attractors:** $K$-random Boolean networks (Kauffman: each gene has $K$
random regulators, random truth tables) have on average $\sqrt{n}$ attractors for $K=2$.

**Perturbation analysis:** Flip one bit in the attractor state. Does the network
return to the same attractor (robust) or escape to another (bifurcation point)?
This defines attractor stability.

**Representation:** Store update rules as truth table functions or lambda expressions.
State as integer (bitmask): gene $i$ is bit $i$ of an integer → efficient.
For $n=20$ genes: $2^{20} = 1{,}048{,}576$ states, easily handled.

In [ ]:
# Step 6 — Python: Boolean network implementation from scratch
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# ---- Boolean network: 6-gene EMT-like model ----
# Genes: E-cad, SNAI1, ZEB1, miR-34, miR-200, GRHL2
# Indices:  0       1      2      3        4        5

# Update rules — encoded as Python lambdas, each taking state tuple
def update_rules(state):
    """
    state: tuple of 0/1 for (E-cad, SNAI1, ZEB1, miR-34, miR-200, GRHL2)
    Returns next state tuple.
    Wiring (simplified EMT network):
      E-cad (0): activated by GRHL2, inhibited by SNAI1, ZEB1
      SNAI1 (1): activated by external signal (assume constant 1 here); inhibited by miR-34
      ZEB1  (2): inhibited by miR-200; activated by SNAI1
      miR-34(3): inhibited by SNAI1; activated by GRHL2
      miR-200(4): inhibited by ZEB1; activated by E-cad
      GRHL2 (5): inhibited by ZEB1; activated by E-cad
    """
    E, S, Z, m34, m200, G = state
    SIGNAL = 1  # external EMT-inducing signal present
    E_next  = int(G and not S and not Z)
    S_next  = int(SIGNAL and not m34)
    Z_next  = int(S and not m200)
    m34_next = int(G and not S)
    m200_next = int(E and not Z)
    G_next  = int(E and not Z)
    return (E_next, S_next, Z_next, m34_next, m200_next, G_next)


# ---- Enumerate all 2^6 = 64 states, simulate to attractor ----
def simulate_to_attractor(initial_state, update_fn, max_steps=100):
    trajectory = [initial_state]
    seen = {initial_state: 0}
    state = initial_state
    for step in range(1, max_steps+1):
        state = update_fn(state)
        if state in seen:
            period = step - seen[state]
            return trajectory, state, period
        seen[state] = step
        trajectory.append(state)
    return trajectory, state, -1  # no attractor found

n_genes = 6
all_states = [tuple(int(b) for b in format(i, f'0{n_genes}b'))
              for i in range(2**n_genes)]

# Find attractors and basin sizes
attractor_map = {}  # state -> attractor state
for state in all_states:
    _, attractor, period = simulate_to_attractor(state, update_rules)
    attractor_map[state] = (attractor, period)

# Count basins
basin_counts = defaultdict(int)
for state, (attr, period) in attractor_map.items():
    basin_counts[(attr, period)] += 1

print('Attractors and basin sizes:')
for (attr, period), count in sorted(basin_counts.items(), key=lambda x: -x[1]):
    names = ['E-cad', 'SNAI1', 'ZEB1', 'miR-34', 'miR-200', 'GRHL2']
    on_genes = [names[i] for i, v in enumerate(attr) if v]
    print(f'  Attractor: {dict(zip(names, attr))} | period={period} | basin={count}/{2**n_genes}')
    print(f'    ON genes: {on_genes}')

In [ ]:
# Simulate time trajectory from two different initial conditions
names = ['E-cad', 'SNAI1', 'ZEB1', 'miR-34', 'miR-200', 'GRHL2']

# IC1: Epithelial (E-cad ON, all others OFF)
ic_epithelial = (1, 0, 0, 0, 0, 0)
traj_epi, attr_epi, _ = simulate_to_attractor(ic_epithelial, update_rules, max_steps=30)

# IC2: Mesenchymal (E-cad OFF, SNAI1 ON, ZEB1 ON)
ic_mesen = (0, 1, 1, 0, 0, 0)
traj_mes, attr_mes, _ = simulate_to_attractor(ic_mesen, update_rules, max_steps=30)

# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: Epithelial trajectory heatmap
ax = axes[0]
mat_epi = np.array(traj_epi)
im = ax.imshow(mat_epi.T, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax.set_yticks(range(n_genes))
ax.set_yticklabels(names, fontsize=8)
ax.set_xlabel('Time step')
ax.set_title(f'A. Epithelial IC → Attractor\n{dict(zip(names, attr_epi))}')
plt.colorbar(im, ax=ax, label='Gene state', shrink=0.7)

# Panel B: Mesenchymal trajectory heatmap
ax = axes[1]
mat_mes = np.array(traj_mes)
im = ax.imshow(mat_mes.T, cmap='Reds', aspect='auto', vmin=0, vmax=1)
ax.set_yticks(range(n_genes))
ax.set_yticklabels(names, fontsize=8)
ax.set_xlabel('Time step')
ax.set_title(f'B. Mesenchymal IC → Attractor\n{dict(zip(names, attr_mes))}')
plt.colorbar(im, ax=ax, label='Gene state', shrink=0.7)

# Panel C: Basin of attraction sizes
ax = axes[2]
basin_items = sorted(basin_counts.items(), key=lambda x: -x[1])
attr_labels = [f'Attr{i+1}\np={item[0][1]}' for i, item in enumerate(basin_items)]
basin_sizes = [item[1] for item in basin_items]
cols = ['steelblue' if basin_items[i][0][0][0] else 'tomato'
          for i in range(len(basin_items))]  # blue=E-cad ON (epithelial), red=E-cad OFF
ax.bar(range(len(basin_items)), basin_sizes, color=cols, edgecolor='black')
ax.set_xticks(range(len(basin_items)))
ax.set_xticklabels(attr_labels, fontsize=8)
ax.set_ylabel('Basin size (# initial conditions)')
ax.set_title(f'C. Basin of attraction sizes\n(blue=E-cad ON, red=E-cad OFF)')
ax.axhline(2**n_genes / len(basin_items), color='grey', ls='--', lw=0.8, label='Random baseline')
ax.legend(fontsize=8)

plt.suptitle('Module 12 NB08: Gene Regulatory Networks — Boolean Model',
               fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('boolean_grn.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Add a 7th gene (TWIST) that activates SNAI1 and is inhibited by E-cad.
   How does this change the number of attractors and basin sizes?
2. Change the SIGNAL to 0 (no EMT-inducing signal). What happens to the SNAI1
   update rule and to the attractor landscape?
3. Implement **asynchronous** update: at each step, pick one gene at random and
   update only that gene. How does this change trajectories?
4. Count the number of attractors in a 10-gene random Boolean network
   (K=2 regulators each). Compare to the Kauffman prediction of ~$\sqrt{n}$.

---
## Step 10 — Quiz

1. What is an attractor in a Boolean network?
2. What is the basin of attraction? What does a large basin mean biologically?
3. How does the number of possible states scale with the number of genes?
4. What is the difference between synchronous and asynchronous update?
5. How do Boolean networks explain bistability in cell fate decisions?

---
## Step 12 — Reflection

> *[In one paragraph: explain why a Boolean network attractor might correspond to
> a cell type, using the EMT example. What biological evidence would support or
> refute this correspondence?]*

---
*Next: `09_ode_models_of_biological_systems.ipynb`*